In [105]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

In [106]:
#carichiamo il csv con i soggetti di Wikidata
df_wikidata = pd.read_csv("wd_dataset.csv", encoding="utf-8")

df_wikidata.head(3)


,entity,label,aliases,description,category
0,http://www.wikidata.org/entity/Q304227,Aba,NaN,ninfa naiade della mitologia greca,greek_deity
1,http://www.wikidata.org/entity/Q279782,Abarbarea,NaN,naiade della mitologia greca,greek_deity
2,http://www.wikidata.org/entity/Q2370052,Acanto,NaN,"personaggio della mitologia greca, probabilmen...",greek_deity


In [107]:
#estraggo tutte le label  
labels = (
    df_wikidata["label"]
    .dropna() #tolgo i valori nulli
    .str.strip() #tolgo gli spazi bianchi all'inizio e alla fine
    .tolist() #converto in lista

)

#estraggo gli aliases separando quelli multipli
aliases = []
for alias_str in df_wikidata["aliases"].dropna(): #per ogni stringa di alias che non è nulla
    alias_list = [alias.strip() for alias in alias_str.split(",")] #separo gli alias con la virgola, tolgo gli spazi e metto tutto in minuscolo
    aliases.extend(alias_list) #aggiungo alla lista totale degli alias

#unisco tutto 
myth_terms = list(set(labels + aliases)) #unisco le label e gli alias, tolgo i duplicati con set e riconverto in lista
len(myth_terms)

3206

In [108]:
#ENDPOINT SPARLQ DI ARCO 
arco_endpoint = "https://dati.cultura.gov.it/sparql"

sparql_arco = SPARQLWrapper(arco_endpoint)
sparql_arco.setReturnFormat(JSON)

In [109]:
# NUMERO TOTALE DEI BENI STORICO ARTISTICI - senza duplicati 
query1 = """
PREFIX arco: <https://w3id.org/arco/ontology/arco/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#> 

SELECT (COUNT(DISTINCT ?item) AS ?totalItems)
WHERE {
  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
}
"""


sparql_arco.setQuery(query1)
results = sparql_arco.query().convert()

for result in results["results"]["bindings"]:
    print(f"Total Historic or Artistic Properties: {result['totalItems']['value']}")

Total Historic or Artistic Properties: 2258640


In [110]:
#query pr estrarre i dati dei beni storico artistici in batch 
# Iterare su tutti i 2.258.640 beni con batch di 20000

rows = []
batch_size = 20000
total_items = 2258640

for offset in range(0, total_items, batch_size):
    print(f"Processando batch: offset={offset}, items processati={len(rows)}")
    
    query2 = f"""
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX a-cd: <https://w3id.org/arco/ontology/context-description/>

SELECT ?item ?subject ?label
WHERE {{

  ?item rdf:type <https://w3id.org/arco/ontology/arco/HistoricOrArtisticProperty> .
  
  OPTIONAL {{ ?item rdfs:label ?label . }}

  OPTIONAL {{ ?item a-cd:subject ?subject . }}

}}
LIMIT {batch_size}
OFFSET {offset}
"""
    
    sparql_arco.setQuery(query2)
    results = sparql_arco.query().convert()
    
    # estrai i dati da questo batch
    for r in results["results"]["bindings"]:
        rows.append({
            "item": r["item"]["value"] if "item" in r else None,
            "label": r["label"]["value"] if "label" in r else None,
            "subject": r["subject"]["value"] if "subject" in r else None
        })
    
    # se il batch è vuoto, ferma il loop
    if len(results["results"]["bindings"]) < batch_size:
        print(f"Batch incompleto ({len(results['results']['bindings'])} items), fine loop")
        break


Processando batch: offset=0, items processati=0
Processando batch: offset=20000, items processati=20000
Processando batch: offset=40000, items processati=40000
Processando batch: offset=60000, items processati=60000
Processando batch: offset=80000, items processati=80000
Processando batch: offset=100000, items processati=100000
Processando batch: offset=120000, items processati=120000
Processando batch: offset=140000, items processati=140000
Processando batch: offset=160000, items processati=160000
Processando batch: offset=180000, items processati=180000
Processando batch: offset=200000, items processati=200000
Processando batch: offset=220000, items processati=220000
Processando batch: offset=240000, items processati=240000
Processando batch: offset=260000, items processati=260000
Processando batch: offset=280000, items processati=280000
Processando batch: offset=300000, items processati=300000
Processando batch: offset=320000, items processati=320000
Processando batch: offset=340000

In [111]:
# i rows sono già stati accumulati dal loop nel batch precedente
# creiamo il dataframe consolidato
df_arco = pd.DataFrame(rows)
df_arco = df_arco.drop_duplicates(subset="item")

print(f"Righe totali in df_arco : {len(df_arco)}")
df_arco[["item", "label", "subject"]].head(10)

Righe totali in df_arco : 1120195


,item,label,subject
0,https://w3id.org/arco/resource/HistoricOrArtis...,Arlecchino primordiale. Fine del cinquecento. ...,"figurino per costume di scena, maschile"
2,https://w3id.org/arco/resource/HistoricOrArtis...,"Pulcinella, maschera della Commedia dell'Arte ...",maschera della Commedia dell'Arte
4,https://w3id.org/arco/resource/HistoricOrArtis...,"Truffaldino, maschera della Commedia dell'Arte...",maschera della Commedia dell'Arte
6,https://w3id.org/arco/resource/HistoricOrArtis...,studio per maschera d'oro (dei re del Gipango)...,studio per maschera d'oro (dei re del Gipango)
8,https://w3id.org/arco/resource/HistoricOrArtis...,"Studio per Medea, figura femminile a mezzo bus...",figura femminile a mezzo busto
10,https://w3id.org/arco/resource/Lombardia/Histo...,Grappolo d'uva (tazzina da caffè) by Ceramiche...,Grappolo d'uva
12,https://w3id.org/arco/resource/HistoricOrArtis...,"uomo seduto con tavola, arcangelo e figure mas...",studio per manifesto
16,https://w3id.org/arco/resource/HistoricOrArtis...,"Regina Isabella, figurino per costume si scena...","figurino per costume si scena, femminile, in s..."
18,https://w3id.org/arco/resource/HistoricOrArtis...,"Arlecchino primordiale, figurino per costume d...","figurino per costume di scena, maschile"
20,https://w3id.org/arco/resource/Lombardia/Histo...,"Kappa su granchio, CREATURA FANTASTICA e ANIMA...",CREATURA FANTASTICA e ANIMALE


In [112]:
import re

# 1. filtro base (toglie null)
df_arco = df_arco[df_arco["subject"].notna()].copy()

terms = [
    re.escape(t)
    for t in myth_terms
    if isinstance(t, str)
    and t.strip() != ""
    and t[0].isupper()
]

# regex unica case-sensitive (matching esatto)
pattern = r'\b(' + '|'.join(terms) + r')\b'
regex = re.compile(pattern)

# matching veloce
df_arco["matched_term"] = df_arco["subject"].str.findall(regex)

# solo match
df_matches = df_arco[df_arco["matched_term"].str.len() > 0]


print(f"Righe con almeno 1 match: {len(df_matches)}")

Righe con almeno 1 match: 20645


In [113]:
# Verifica i risultati
print(f"Totale beni Arco: {len(df_arco)}")
print(f"Beni con match mitologico: {len(df_matches)}")

Totale beni Arco: 522845
Beni con match mitologico: 20645


In [125]:
start = 200
end = 300

df_matches_sorted = df_matches.sort_values("matched_term")

df_matches_sorted[["label", "subject", "matched_term"]].iloc[start:end]


,label,subject,matched_term
749596,"Adone (disegno, opera isolata) by Boucheron An...",Adone,[Adone]
125034,Nacita di Adone (rilievo) by Thorvaldsen Berte...,Nacita di Adone,[Adone]
1076062,morte di Adone (dipinto) di Zucchi Jacopo (ult...,morte di Adone,[Adone]
178412,"Adone (statua, opera isolata) di Campagna Giro...",Adone,[Adone]
688686,morte di Adone (scultura) di Rossi Vincenzo de...,morte di Adone,[Adone]
645364,Morte di Adone (stampa smarginata) by Penni Lo...,Morte di Adone,[Adone]
1070482,Due putti che si sporgono/ Adone (dipinto) di ...,Due putti che si sporgono/ Adone,[Adone]
688942,morte di Adone (dipinto) by Luciani Sebastiano...,morte di Adone,[Adone]
1164660,"Adone cacciatore (orologio - da mensola, opera...",Adone cacciatore,[Adone]
1054972,"Adone (statua, opera isolata) di Parodi Filipp...",Adone,[Adone]
